# Text Preprocessing and TF-IDF Feature Engineering

This notebook explores and demonstrates the text preprocessing pipeline and TF-IDF vectorization.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)

Path('../results/figures').mkdir(parents=True, exist_ok=True)

## 1. Load and Split Data

In [ ]:
# Load data
loader = DataLoader('../config/config.yaml')
dataset = loader.load_dataset()
df = loader.create_dataframe(dataset)
train_df, val_df, test_df = loader.split_train_validation(df)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## 2. Text Cleaning Demonstration

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor('../config/config.yaml')

# Show raw vs cleaned text examples
print("=== RAW vs CLEANED TEXT EXAMPLES ===\n")

for i in range(3):
    raw = train_df.iloc[i]['document_text']
    cleaned = preprocessor.clean_text(raw)
    print(f"--- Example {i+1} ---")
    print(f"RAW (first 500 chars):")
    print(raw[:500])
    print(f"\nCLEANED (first 500 chars):")
    print(cleaned[:500])
    print("\n" + "="*80 + "\n")

## 3. Apply Preprocessing to All Splits

In [ ]:
# Clean all splits
train_df = preprocessor.preprocess_dataframe(train_df)
val_df = preprocessor.preprocess_dataframe(val_df)
test_df = preprocessor.preprocess_dataframe(test_df)

# Check cleaning stats
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    empty = (df['cleaned_text'].str.strip() == '').sum()
    avg_len = df['cleaned_text'].str.len().mean()
    print(f"{name}: {len(df)} docs, {empty} empty, avg length: {avg_len:.0f} chars")

## 4. TF-IDF Vectorization

In [ ]:
# Fit vectorizer on training data only
X_train = preprocessor.fit_transform(train_df['cleaned_text'].tolist())
X_val = preprocessor.transform_texts(val_df['cleaned_text'].tolist())
X_test = preprocessor.transform_texts(test_df['cleaned_text'].tolist())

print(f"Training matrix shape: {X_train.shape}")
print(f"Validation matrix shape: {X_val.shape}")
print(f"Test matrix shape: {X_test.shape}")
print(f"Vocabulary size: {preprocessor.get_vocabulary_size()}")
print(f"Matrix density: {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.4f}%")

## 5. Label Encoding

In [ ]:
# Fit label encoder
preprocessor.fit_label_encoder(train_df['category'].tolist())

y_train = preprocessor.encode_labels(train_df['category'].tolist())
y_val = preprocessor.encode_labels(val_df['category'].tolist())
y_test = preprocessor.encode_labels(test_df['category'].tolist())

print(f"Training labels shape: {y_train.shape}")
print(f"Validation labels shape: {y_val.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Number of classes: {len(preprocessor.label_encoder.classes_)}")
print(f"Classes: {preprocessor.label_encoder.classes_}")

## 6. TF-IDF Feature Analysis

In [ ]:
# Analyze TF-IDF features
feature_names = preprocessor.get_feature_names()

# Get IDF values
idf_values = preprocessor.vectorizer.idf_

# Top features by IDF (most discriminative)
top_idf_idx = np.argsort(idf_values)[-20:][::-1]
print("=== TOP 20 FEATURES BY IDF (most discriminative) ===")
for idx in top_idf_idx:
    print(f"  {feature_names[idx]}: IDF={idf_values[idx]:.4f}")

# Bottom features by IDF (least discriminative)
bottom_idf_idx = np.argsort(idf_values)[:20]
print("\n=== BOTTOM 20 FEATURES BY IDF (least discriminative) ===")
for idx in bottom_idf_idx:
    print(f"  {feature_names[idx]}: IDF={idf_values[idx]:.4f}")

In [ ]:
# IDF distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(idf_values, bins=50, color='#2E86C1', alpha=0.8, edgecolor='white')
axes[0].axvline(idf_values.mean(), color='red', linestyle='--', label=f'Mean: {idf_values.mean():.2f}')
axes[0].set_xlabel('IDF Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('IDF Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# TF-IDF matrix sparsity pattern (sample)
sample_matrix = X_train[:100, :500]
axes[1].spy(sample_matrix, markersize=0.5, color='#E74C3C')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Document Index')
axes[1].set_title('TF-IDF Sparsity Pattern (100 docs × 500 features)')

plt.tight_layout()
plt.savefig('../results/figures/tfidf_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 7. Category-wise TF-IDF Analysis

In [ ]:
# Average TF-IDF per category for top features
import scipy.sparse as sp

# Get top 50 features by overall TF-IDF sum
feature_sums = np.array(X_train.sum(axis=0)).flatten()
top_feature_idx = np.argsort(feature_sums)[-50:][::-1]

# Compute mean TF-IDF per category for these features
cat_names = preprocessor.label_encoder.classes_
cat_feature_means = {}

for i, cat in enumerate(cat_names):
    cat_mask = (y_train == i)
    if cat_mask.sum() > 0:
        cat_means = X_train[cat_mask][:, top_feature_idx].mean(axis=0)
        cat_feature_means[cat] = np.array(cat_means).flatten()

# Create heatmap data
heatmap_data = pd.DataFrame(cat_feature_means, index=[feature_names[i] for i in top_feature_idx])
heatmap_data = heatmap_data.T  # Categories as rows

# Plot
plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Mean TF-IDF'})
plt.title('Mean TF-IDF for Top 50 Features by Category')
plt.xlabel('Features')
plt.ylabel('Category')
plt.tight_layout()
plt.savefig('../results/figures/tfidf_category_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Save Preprocessing Artifacts

In [ ]:
# Save artifacts
preprocessor.save_artifacts()

# Save sparse matrices
from scipy.sparse import save_npz
import numpy as np

save_npz('../data/processed/X_train.npz', X_train)
save_npz('../data/processed/X_val.npz', X_val)
save_npz('../data/processed/X_test.npz', X_test)

np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)

print("All preprocessing artifacts saved successfully!")

## 9. Preprocessing Summary

### Preprocessing Pipeline Summary:

1. **Text Cleaning**:
   - Lowercasing
   - Email/URL removal
   - Newsgroup header removal (From, Subject, Organization, etc.)
   - Quoted text removal (lines starting with > or |)
   - Whitespace normalization

2. **TF-IDF Vectorization**:
   - Max features: 15,000
   - N-gram range: (1, 2) - unigrams + bigrams
   - Min DF: 2, Max DF: 0.95
   - Sublinear TF: True
   - Stop words: English
   - Token pattern: Words with 2+ letters
   - Fitted ONLY on training data

3. **Label Encoding**:
   - LabelEncoder fitted on training categories
   - 20 classes mapped to integers 0-19

4. **Artifacts Saved**:
   - `tfidf_vectorizer.joblib` - fitted vectorizer
   - `label_encoder.joblib` - fitted label encoder
   - `X_train.npz`, `X_val.npz`, `X_test.npz` - sparse TF-IDF matrices
   - `y_train.npy`, `y_val.npy`, `y_test.npy` - encoded labels